# SmartVal: Why Hidden Layers and Backpropagation Exist

> **What you already know:** train/validation/test rows have separate jobs, a metric is not a loss, and capacity must be judged on held-out evidence.
> **The gap this chapter closes:** SmartVal's linear gate cannot express a decision whose positive cases sit on opposite corners.
> **What you'll have by the end:** a visible XOR representation, a four-weight responsibility report, and a numerical gradient audit.
> **What this chapter is not:** another full training-loop tutorial; the canonical PyTorch route owns implementation mechanics.

**Guiding question:** What information must a network create before one straight decision boundary can solve XOR?

| Attempt | Visible complaint | Smallest refinement |
|---|---|---|
| one weighted sum | XOR is not linearly separable | create hidden features |
| hidden features | predictions improve, but blame is unclear | propagate responsibility backward |
| analytic blame | formulas can be implemented incorrectly | compare with finite differences |

\`\`\`mermaid
flowchart LR
    A["Two binary inputs"] --> B["Linear score"]
    B --> C["Failure: opposite corners"]
    C --> D["Two ReLU features"]
    D --> E["Linear output succeeds"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Imports and the four SmartVal XOR cases ---------------------------------
import matplotlib.pyplot as plt
import numpy as np

X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = np.array([[0.], [1.], [1.], [0.]])
print(f"Input contract: X={X.shape} (case, signal), y={y.shape} (case, decision)")


## 1. Attempt 1: one straight boundary

**Predict:** Can any line put \`(0, 1)\` and \`(1, 0)\` on one side while keeping \`(0, 0)\` and \`(1, 1)\` on the other?

Rather than assert "no," search a grid of linear classifiers and report the best accuracy.


In [ ]:
# -- Exhaust a visible grid of linear boundaries ------------------------------
best = (0.0, None)
for w1 in np.linspace(-3, 3, 49):
    for w2 in np.linspace(-3, 3, 49):
        for bias in np.linspace(-3, 3, 49):
            prediction = ((X @ np.array([[w1], [w2]]) + bias) >= 0).astype(float)
            accuracy = float((prediction == y).mean())
            if accuracy > best[0]:
                best = (accuracy, (w1, w2, bias, prediction.ravel()))

print(f"Best searched linear accuracy: {best[0]:.0%}")
print(f"One best prediction pattern: {best[1][3].astype(int).tolist()}")
print("Complaint: one affine score cannot isolate opposite corners.")


In [ ]:
# -- Static geometry: see why one line is the wrong tool ----------------------
fig, ax = plt.subplots(figsize=(5, 5))
for label, marker in ((0, "o"), (1, "s")):
    points = X[y.ravel() == label]
    ax.scatter(points[:, 0], points[:, 1], s=160, marker=marker, label=f"target {label}")
for point, label in zip(X, y.ravel()):
    ax.annotate(f" {int(label)}", point)
ax.set(xlabel="signal A", ylabel="signal B", title="XOR keeps equal labels on opposite corners")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("The plot and grid search expose the same limitation.")


## 2. Attempt 2: make the missing representation

Two hidden units can ask different questions:

\`\`\`text
OR-like feature:   ReLU(x1 + x2)
AND-like feature:  ReLU(x1 + x2 - 1)
XOR score:         OR - 2 * AND
\`\`\`

The output is linear in hidden space. The nonlinearity changed the representation, not the four labels.

\`\`\`mermaid
flowchart LR
    A["x1, x2"] --> B["h1: any signal?"]
    A --> C["h2: both signals?"]
    B --> D["h1 - 2h2"]
    C --> D
    D --> E["XOR decision"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Build a readable hidden representation with fixed weights ----------------
W_hidden = np.array([[1., 1.], [1., 1.]])
b_hidden = np.array([[0., -1.]])
W_out = np.array([[1.], [-2.]])
b_out = np.array([[0.]])

hidden_pre = X @ W_hidden + b_hidden
hidden = np.maximum(hidden_pre, 0.0)
score = hidden @ W_out + b_out
prediction = (score > 0).astype(float)

print("case -> hidden coordinates -> score -> prediction")
for x_row, h_row, score_row, pred_row in zip(X, hidden, score, prediction):
    print(f"{x_row.astype(int).tolist()} -> {h_row.tolist()} -> {score_row[0]:.1f} -> {int(pred_row[0])}")
print(f"Hidden representation accuracy: {(prediction == y).mean():.0%}")


In [ ]:
# -- Show that the hidden coordinates make a straight split possible ----------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", s=140)
axes[0].set(title="Input space", xlabel="x1", ylabel="x2")
axes[1].scatter(hidden[:, 0], hidden[:, 1], c=y.ravel(), cmap="coolwarm", s=140)
axes[1].set(title="Hidden space", xlabel="h1: any", ylabel="h2: both")
for axis in axes:
    axis.grid(alpha=0.3)
plt.show()
print("Measured payoff: the same four cases are now linearly readable.")


### That worked, but which weight was responsible?

A forward pass answers "what did the model predict?" Backpropagation answers a different question: "If the loss changed a little, how much responsibility reaches each earlier quantity?"

\`\`\`mermaid
flowchart RL
    L["Loss"] --> O["Output score"]
    O --> W2["Output weights"]
    O --> H["Hidden activations"]
    H --> R["ReLU gates"]
    R --> W1["Input weights"]
    style L fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W2 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W1 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- One-case forward pass and manual responsibility report -------------------
x_case = np.array([[1.0, 0.0]])
target = np.array([[1.0]])
W1 = np.array([[0.8, -0.4], [0.3, 0.9]])
b1 = np.array([[0.1, -0.2]])
W2 = np.array([[0.7], [-0.5]])
b2 = np.array([[0.05]])

z1 = x_case @ W1 + b1
h1 = np.maximum(z1, 0.0)
output = h1 @ W2 + b2
loss = 0.5 * (output - target) ** 2

d_output = output - target
dW2 = h1.T @ d_output
dh1 = d_output @ W2.T
dz1 = dh1 * (z1 > 0)
dW1 = x_case.T @ dz1

print(f"Forward: x{x_case.shape} -> z{z1.shape} -> h{h1.shape} -> output{output.shape}")
print(f"Loss={loss.item():.6f}")
print("Responsibility report dW1:")
print(dW1)


## 3. A formula is not evidence: audit the gradients

Finite differences disturb one weight by a tiny amount and observe the loss change. It is slower than backpropagation but independent enough to catch a chain-rule bug.

**Predict:** Should the two reports be exactly equal? No. They should agree within the approximation error created by the small disturbance.


In [ ]:
# -- Compare analytic and finite-difference gradients -------------------------
def loss_for_W1(candidate_W1):
    candidate_z = x_case @ candidate_W1 + b1
    candidate_h = np.maximum(candidate_z, 0.0)
    candidate_output = candidate_h @ W2 + b2
    return (0.5 * (candidate_output - target) ** 2).item()

epsilon = 1e-5
finite_difference = np.zeros_like(W1)
for row in range(W1.shape[0]):
    for column in range(W1.shape[1]):
        plus = W1.copy()
        minus = W1.copy()
        plus[row, column] += epsilon
        minus[row, column] -= epsilon
        finite_difference[row, column] = (
            loss_for_W1(plus) - loss_for_W1(minus)
        ) / (2 * epsilon)

max_error = np.max(np.abs(dW1 - finite_difference))
print("Finite-difference report:")
print(finite_difference)
print(f"Maximum absolute disagreement: {max_error:.2e}")
assert max_error < 1e-7


## 4. Depth, width, dropout, and normalization: concise boundaries

These tools solve different complaints:

| Tool | What it changes | What it does not guarantee |
|---|---|---|
| width | number of features at one stage | useful features or generalization |
| depth | number of composed transformations | easy optimization |
| dropout | randomly removes activations during training | calibrated uncertainty |
| normalization | controls activation scale by a stated axis | correct targets or clean data |

Dropout creates a train/evaluation distinction. A deterministic NumPy mask is enough to prove the scaling contract without introducing a framework training loop.


In [ ]:
# -- Measure dropout's train/evaluation contract ------------------------------
rng = np.random.default_rng(5)
activation = np.ones((10_000,))
drop_probability = 0.4
mask = rng.random(activation.shape) >= drop_probability
training_activation = activation * mask / (1 - drop_probability)
evaluation_activation = activation

print(f"Training mean after inverted dropout: {training_activation.mean():.3f}")
print(f"Evaluation mean with no dropping: {evaluation_activation.mean():.3f}")
print("The scaling preserves expectation, not each individual activation.")


### Your turn: close one ReLU gate


In [ ]:
# -- CHANGE THIS: make one hidden pre-activation negative ----------------------
YOUR_BIAS_SHIFT = -1.0
changed_z = z1.copy()
changed_z[0, 0] += YOUR_BIAS_SHIFT
changed_gate = (changed_z > 0).astype(int)
changed_gradient = dh1 * changed_gate
print(f"Changed pre-activation: {changed_z.ravel().tolist()}")
print(f"ReLU gates: {changed_gate.ravel().tolist()}")
print(f"Responsibility reaching the hidden units: {changed_gradient.ravel().tolist()}")


## Scorecard and coverage ledger

| Opening question | Evidence |
|---|---|
| Why is a hidden layer necessary? | the searched linear boundary stopped below perfect XOR accuracy |
| What did the hidden layer create? | two coordinates that made all four cases linearly readable |
| What does backpropagation report? | loss sensitivity for each earlier weight |
| Why trust the report? | analytic and finite-difference gradients agreed within the printed tolerance |

| Tier | Covered here |
|---|---|
| Built and measured | XOR failure, hidden representation, manual chain rule, finite-difference audit, dropout expectation |
| Explained or illustrated | depth, width, dropout, normalization boundaries |
| Named and out of scope | full optimizer loop, autograd APIs, convolution, recurrent state, production initialization |

**Common failure modes:** treating ReLU as decorative, reporting a gradient without checking its shape, forgetting that inactive ReLU units pass zero local gradient, and comparing models while changing several conditions.

**Optional audible capstone:** [Melodyne Backprop Lab](optional/02-melodyne-backprop-synthesizer.ipynb) keeps differentiable rendering, four-gradient auditing, recovery gates, and audio evidence outside the required route.

**Forward:** you now know why gradients exist. Next, the canonical PyTorch route makes tensor state, modules, optimizers, data loaders, and checkpoints explicit.
